In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


/Users/rutwikbelsare/Rutwik/Gen AI/Projects/RagProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
##Intialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

#Sample Text
text = """
    LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

#step 1: Split into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

#step2: Embed each sentence
embeddings = model.encode(sentences)

##Step 3 : Initilaize Paramenters
threshold = 0.7
chunks=[]
current_chunk = [sentences[0]]

##step 4: Semantic Grouping based on the threshold

for i in range(1,len(sentences)):
    sim=cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

#Append the last chunk
chunks.append(" ".join(current_chunk))

#output the chunks

print("\nSemantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"Chunks {idx+1}:\n{chunk}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6748.84it/s]



Semantic Chunks:
Chunks 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunks 2:
You can create chains, agents, memory, and retrievers.
Chunks 3:
The Eiffel Tower is located in Paris.
Chunks 4:
France is a popular tourist destination.


## RAG Pipeline with Modular Coding

In [ ]:
from sentence_transformers import SentenceTransformer #Used for chunking
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings #Used to convert chunks into vecotr embeddings
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [11]:
class SemanticThresholdChunker:
    def __init__(self, model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
), threshold = 0.7):
        self.model = model
        self.threshold = threshold

    def split(self, text:str):
        ##removing white spaces
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        #converting to embeddings:
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
            if sim>=self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
        chunks.append(". ".join(current_chunk) + ".")
        return chunks

    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata = doc.metadata))
        return result
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7077.08it/s]


In [12]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [13]:
##Chunking
chunker = SemanticThresholdChunker(threshold = 0.7)
chunks = chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs.. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone..'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers..'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris..'),
 Document(metadata={}, page_content='France is a popular tourist destination..')]

In [15]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(chunks,embeddings)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6455.42it/s]


In [16]:
##prompt template
template = """Answere the question based on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answere the question based on the following context:\n{context}\n\nQuestion: {question}\n')

In [20]:
##LLM
from langchain.chat_models import init_chat_model
llm=init_chat_model("openai/gpt-oss-120b",model_provider="groq")

##LCEL Chain with retrieval
rag_chain=(
    RunnableMap(
        {
            "context":lambda x : retriever.invoke(x["question"]),
            "question": lambda x: x['question'],
        }
    )
    |prompt
    |llm
    |StrOutputParser()

)

In [21]:
#____8. Query----------
query = {"question":"what is langchain used for?"}
result = rag_chain.invoke(query)
result

'LangChain is a framework for building applications that use large language models (LLMs). It provides modular abstractions that let you combine LLMs with external tools (e.g., OpenAI, Pinecone) and create components such as chains, agents, memory modules, and retrievers. In short, LangChain is used to construct LLM‑driven applications and workflows.'

## Semantic Chunking with LANGCHAIN

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

#Load the documents
loader = TextLoader('langchain_intro.txt')
docs = loader.load()
#Initialize Embedding Models
embeddings = HuggingFaceEmbeddings()

#Create Semantic Chunker
chunker = SemanticChunker(embeddings)

##Split the documents

chunks = chunker.split_documents(docs)

##Result
for i, chunk in enumerate(chunks):
    print(f"Chunk: {i+1}, \n {chunk.page_content}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5083.94it/s]


Chunk: 1, 
 LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.
Chunk: 2, 
 The Eiffel Tower is located in Paris. France is a popular tourist destination.
